In [14]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os
from langchain_google_genai import ChatGoogleGenerativeAI

In [15]:
load_dotenv(override=True)

# Initialize Gemini Model
api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
# Generator: Production model for complex creative/technical output
llm = ChatGoogleGenerativeAI(
    model='gemini-3.6-flash',  # gemini-3.6-flash
    max_retries=6,
    google_api_key=api_key
)

In [16]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [17]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [18]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [19]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [20]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'What do you call a fake pizza?\n\nA **pepper-phony**!',
   'extras': {'signature': 'Es4YCssYARFNMg+Mzeai0DP/+XMl8yET+Ze3aGPPHwxpuioOw+pR5mPrrGQ9Rlq9rPORl2OXn2mloptbOh1w5xQ+dO6w70sGSFH+JwAEXbka7JRNsqo2eEYeuoaGUjvUzC7V8xH9miQbBDzOmk7Bb6un9VA/3LdAOW8OAR/I9kIBpCa6AS0h4Vk5PEPIxEBaf7fQvLmZYIdXP4KImcM1orOyam4nKhXJ2huo0Dv7e1QpMM6AjG3YKfnn+m6SKXDFtUj/g8BECbaJRNzbU2Kc7YPtfsTOIEXl9y25M14u2SLfxM9hIu4SYW77cbvWs9pUN0QIn2AR+tgHdbZpHBiB6akMBX5CjhrwlRY4gJkyrQla6GXt4kVcgOyC9ivkqZ8wXJHI2lTrc2G+s7klZrFmO9pp/snCCxPQxp5WdF6J8ASHe6KAjW1kvzbRRVrSEnX/4aCcrqQh/yQMBfIDE3Mrm2FN8pcL1GSEC5Il88O4k8BQQHkETlIUrZVh5IOJAfxNQ431pOBUGDnWU2VgrOBmUnQYKi9/j3U/8DYP/NiDA/moSXylWhwukLTbaUHUa27MFrHKDPEEx4ssK+Fy4IyakGRdOiYwqb4QenikUbuIiLIkDYCgks1OiBz7iOcsFOMxKA1iVpnAmr+nRETMwjvmF+A/6Pam/mWEtqDv585aV5ikrvirn1iQL+UDPD5IdZC80BWOMvb4kgc15k6Gde4xyDF2kTllVLGvbdSedoaJ2w11+yVPUSW2IGzCw46ogsyM+h/8my74kjv2ZWA1PehLYSI2koszooBXWLBzCM7j4K9TjjihPAPH8EBG1S5D/rM7m0y5jaIvNIWD/XVqrUxehPd4wf

In [21]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What do you call a fake pizza?\n\nA **pepper-phony**!', 'extras': {'signature': 'Es4YCssYARFNMg+Mzeai0DP/+XMl8yET+Ze3aGPPHwxpuioOw+pR5mPrrGQ9Rlq9rPORl2OXn2mloptbOh1w5xQ+dO6w70sGSFH+JwAEXbka7JRNsqo2eEYeuoaGUjvUzC7V8xH9miQbBDzOmk7Bb6un9VA/3LdAOW8OAR/I9kIBpCa6AS0h4Vk5PEPIxEBaf7fQvLmZYIdXP4KImcM1orOyam4nKhXJ2huo0Dv7e1QpMM6AjG3YKfnn+m6SKXDFtUj/g8BECbaJRNzbU2Kc7YPtfsTOIEXl9y25M14u2SLfxM9hIu4SYW77cbvWs9pUN0QIn2AR+tgHdbZpHBiB6akMBX5CjhrwlRY4gJkyrQla6GXt4kVcgOyC9ivkqZ8wXJHI2lTrc2G+s7klZrFmO9pp/snCCxPQxp5WdF6J8ASHe6KAjW1kvzbRRVrSEnX/4aCcrqQh/yQMBfIDE3Mrm2FN8pcL1GSEC5Il88O4k8BQQHkETlIUrZVh5IOJAfxNQ431pOBUGDnWU2VgrOBmUnQYKi9/j3U/8DYP/NiDA/moSXylWhwukLTbaUHUa27MFrHKDPEEx4ssK+Fy4IyakGRdOiYwqb4QenikUbuIiLIkDYCgks1OiBz7iOcsFOMxKA1iVpnAmr+nRETMwjvmF+A/6Pam/mWEtqDv585aV5ikrvirn1iQL+UDPD5IdZC80BWOMvb4kgc15k6Gde4xyDF2kTllVLGvbdSedoaJ2w11+yVPUSW2IGzCw46ogsyM+h/8my74kjv2ZWA1PehLYSI2koszooBXWLBzCM7j4K9TjjihPAPH8EBG1S5D/rM7m0y5jaIvNIWD

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What do you call a fake pizza?\n\nA **pepper-phony**!', 'extras': {'signature': 'Es4YCssYARFNMg+Mzeai0DP/+XMl8yET+Ze3aGPPHwxpuioOw+pR5mPrrGQ9Rlq9rPORl2OXn2mloptbOh1w5xQ+dO6w70sGSFH+JwAEXbka7JRNsqo2eEYeuoaGUjvUzC7V8xH9miQbBDzOmk7Bb6un9VA/3LdAOW8OAR/I9kIBpCa6AS0h4Vk5PEPIxEBaf7fQvLmZYIdXP4KImcM1orOyam4nKhXJ2huo0Dv7e1QpMM6AjG3YKfnn+m6SKXDFtUj/g8BECbaJRNzbU2Kc7YPtfsTOIEXl9y25M14u2SLfxM9hIu4SYW77cbvWs9pUN0QIn2AR+tgHdbZpHBiB6akMBX5CjhrwlRY4gJkyrQla6GXt4kVcgOyC9ivkqZ8wXJHI2lTrc2G+s7klZrFmO9pp/snCCxPQxp5WdF6J8ASHe6KAjW1kvzbRRVrSEnX/4aCcrqQh/yQMBfIDE3Mrm2FN8pcL1GSEC5Il88O4k8BQQHkETlIUrZVh5IOJAfxNQ431pOBUGDnWU2VgrOBmUnQYKi9/j3U/8DYP/NiDA/moSXylWhwukLTbaUHUa27MFrHKDPEEx4ssK+Fy4IyakGRdOiYwqb4QenikUbuIiLIkDYCgks1OiBz7iOcsFOMxKA1iVpnAmr+nRETMwjvmF+A/6Pam/mWEtqDv585aV5ikrvirn1iQL+UDPD5IdZC80BWOMvb4kgc15k6Gde4xyDF2kTllVLGvbdSedoaJ2w11+yVPUSW2IGzCw46ogsyM+h/8my74kjv2ZWA1PehLYSI2koszooBXWLBzCM7j4K9TjjihPAPH8EBG1S5D/rM7m0y5jaIvNIW

In [23]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'What do you call pasta that’s pretending to be a detective?\n\nAn **impasta** investigating a cold sauce!',
   'extras': {'signature': 'EqQlCqElARFNMg9aeoY8B15EyBhBsAk5lDs5WotgVZPUS9SfZLMwu+KRhlUJOWppdHzCHi0kEtjFjuY+y9CCjwt3FLp1EO9TQd3ejYd/r68B7MwP6Z8mGa0WgjJsi+03zYJNaU5MzA/VqpeatuBWnsD9LMGIAXSgKaMML//+1jEBiXJ6tJkNxDgp65DKWje7t3xGivSNBfNn+NcTItVgjmx+F0oXIRurojDNsAom+AGKi/pTYZ536ZLOJLUpXe4sbiWmQgdsJVYjVVLOnimEcuGNMVjbuUExURFKhHxP0VOkd7jm//+T3JqUQW9U26DVSUaD++rDekZh59LX8ycfYRCbSbBcYTLv4HY7Cgq/slWZOgyqi8NH4Wy7pgaCGbrDIpebRzShLwW8pIcR0pC+6LCDl02jq/NMsjUVxCVoR+J9tEtJTDEETkYJf8Ia5RxGI22NezG5lnwqNzjCG8LP/kIh7voJC9h04AU0xCeGzLIP7WSBIsLcfdiSZev5HfoXfZL/RW8T4CGh4XMaLysNcHr8W1mcY7t05PT8EMVA6vBYKfZv7VqjV/RZVT815qeASED0/FIhG+QC76XBUrbEJhsME0z5jFJT0zW82taVCt0ZouXMSQTeVV5Ekvg6mKURVUioZ1NRQ12M0aa4/gGcB1bCd1uNzjz/9UXr7fFCTYMbvpiLE2Sl2rKP1dOR2cQSjNpICDQZ66jk9d/26sSH551G3eB8TQhGpJe0L631sUzdZsu5WpuTnBFa+mEyWEfJw8JAvnLPo3KyMnDa1SrAu9TP9YqRfZcOsQrJcC

In [24]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What do you call a fake pizza?\n\nA **pepper-phony**!', 'extras': {'signature': 'Es4YCssYARFNMg+Mzeai0DP/+XMl8yET+Ze3aGPPHwxpuioOw+pR5mPrrGQ9Rlq9rPORl2OXn2mloptbOh1w5xQ+dO6w70sGSFH+JwAEXbka7JRNsqo2eEYeuoaGUjvUzC7V8xH9miQbBDzOmk7Bb6un9VA/3LdAOW8OAR/I9kIBpCa6AS0h4Vk5PEPIxEBaf7fQvLmZYIdXP4KImcM1orOyam4nKhXJ2huo0Dv7e1QpMM6AjG3YKfnn+m6SKXDFtUj/g8BECbaJRNzbU2Kc7YPtfsTOIEXl9y25M14u2SLfxM9hIu4SYW77cbvWs9pUN0QIn2AR+tgHdbZpHBiB6akMBX5CjhrwlRY4gJkyrQla6GXt4kVcgOyC9ivkqZ8wXJHI2lTrc2G+s7klZrFmO9pp/snCCxPQxp5WdF6J8ASHe6KAjW1kvzbRRVrSEnX/4aCcrqQh/yQMBfIDE3Mrm2FN8pcL1GSEC5Il88O4k8BQQHkETlIUrZVh5IOJAfxNQ431pOBUGDnWU2VgrOBmUnQYKi9/j3U/8DYP/NiDA/moSXylWhwukLTbaUHUa27MFrHKDPEEx4ssK+Fy4IyakGRdOiYwqb4QenikUbuIiLIkDYCgks1OiBz7iOcsFOMxKA1iVpnAmr+nRETMwjvmF+A/6Pam/mWEtqDv585aV5ikrvirn1iQL+UDPD5IdZC80BWOMvb4kgc15k6Gde4xyDF2kTllVLGvbdSedoaJ2w11+yVPUSW2IGzCw46ogsyM+h/8my74kjv2ZWA1PehLYSI2koszooBXWLBzCM7j4K9TjjihPAPH8EBG1S5D/rM7m0y5jaIvNIWD

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What do you call a fake pizza?\n\nA **pepper-phony**!', 'extras': {'signature': 'Es4YCssYARFNMg+Mzeai0DP/+XMl8yET+Ze3aGPPHwxpuioOw+pR5mPrrGQ9Rlq9rPORl2OXn2mloptbOh1w5xQ+dO6w70sGSFH+JwAEXbka7JRNsqo2eEYeuoaGUjvUzC7V8xH9miQbBDzOmk7Bb6un9VA/3LdAOW8OAR/I9kIBpCa6AS0h4Vk5PEPIxEBaf7fQvLmZYIdXP4KImcM1orOyam4nKhXJ2huo0Dv7e1QpMM6AjG3YKfnn+m6SKXDFtUj/g8BECbaJRNzbU2Kc7YPtfsTOIEXl9y25M14u2SLfxM9hIu4SYW77cbvWs9pUN0QIn2AR+tgHdbZpHBiB6akMBX5CjhrwlRY4gJkyrQla6GXt4kVcgOyC9ivkqZ8wXJHI2lTrc2G+s7klZrFmO9pp/snCCxPQxp5WdF6J8ASHe6KAjW1kvzbRRVrSEnX/4aCcrqQh/yQMBfIDE3Mrm2FN8pcL1GSEC5Il88O4k8BQQHkETlIUrZVh5IOJAfxNQ431pOBUGDnWU2VgrOBmUnQYKi9/j3U/8DYP/NiDA/moSXylWhwukLTbaUHUa27MFrHKDPEEx4ssK+Fy4IyakGRdOiYwqb4QenikUbuIiLIkDYCgks1OiBz7iOcsFOMxKA1iVpnAmr+nRETMwjvmF+A/6Pam/mWEtqDv585aV5ikrvirn1iQL+UDPD5IdZC80BWOMvb4kgc15k6Gde4xyDF2kTllVLGvbdSedoaJ2w11+yVPUSW2IGzCw46ogsyM+h/8my74kjv2ZWA1PehLYSI2koszooBXWLBzCM7j4K9TjjihPAPH8EBG1S5D/rM7m0y5jaIvNIW

### Time Travel

In [26]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f19aea9-fff9-6e08-8000-ce7aa18cf48f"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19aea9-fff9-6e08-8000-ce7aa18cf48f"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza chef go to therapy?\n\nBecause he felt like nobody **kneaded** him, and everybody just wanted a **piece** of him!',
   'extras': {'signature': 'ErYjCrMjARFNMg/o16DTJFitEq+L53843Soo6Rtc7o2a7sBSbZflTAKfaz4Mu/Y72VbYlaOxnQTv84AKUyIhpC10b7V4Nr8gDMEHCnWWyBg7WAH4L0ETSOxW0c14hD/5HND7u6VfFXEfG92pO3rHG467A96U6qn2gNr6LuDlDekShun65K/A/Csvan0vFeDIMgFFh/HTU+QvjLHVT2b9GL3XBbUFRkNC3bthqq9MpxIkzT/MxrutE7ZA+zBmg8OboXA+Pg4elSUuOIQdbQSo8yfU3kIu+iLV+DCjvLvoALCT00GaqgidPygRFKwwudIxu3RmUbldiu51fB4vLQVnGHL65MM2RmpB9C9RXYmwxCdlO3GnCuR6bJzKpalwfNnGfz5a30uL7S7Wvli8i/u2shOaJN+NPQR0+DIUiBlQFBvlsqT4OvszfQu1nJCNBqTE33uI7QIZ5HKneGpURb6DaQGuRP7gEyRMvhXrUZ/hW84ASXqpwdANJ/Q9zDd4xuOYxkNIfHa8cvnYaVIGfP+2Wfh7u8z96yYSkHt/4pt1Qp5Ep9Z/8XB0GMRpxKco16wx9wF1EMMjbSCTBx8is6D2UCsRCKotrA1KRbRvhRHF0y/BkAX6Wb0EvaY1Lh21dx/9krublD1r/+xCd1sWzlCtIyPmVCaVmizNiBOLz+wtrnvPSSg4chpgq0YALjH5TSWYIwIhvrwUmiEOgz69LF7jSNAqZ43ERAqtR692VAP7Os/O7CDykYINcEkFh9vjP4xsjet7TqWt/jMW

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza chef go to therapy?\n\nBecause he felt like nobody **kneaded** him, and everybody just wanted a **piece** of him!', 'extras': {'signature': 'ErYjCrMjARFNMg/o16DTJFitEq+L53843Soo6Rtc7o2a7sBSbZflTAKfaz4Mu/Y72VbYlaOxnQTv84AKUyIhpC10b7V4Nr8gDMEHCnWWyBg7WAH4L0ETSOxW0c14hD/5HND7u6VfFXEfG92pO3rHG467A96U6qn2gNr6LuDlDekShun65K/A/Csvan0vFeDIMgFFh/HTU+QvjLHVT2b9GL3XBbUFRkNC3bthqq9MpxIkzT/MxrutE7ZA+zBmg8OboXA+Pg4elSUuOIQdbQSo8yfU3kIu+iLV+DCjvLvoALCT00GaqgidPygRFKwwudIxu3RmUbldiu51fB4vLQVnGHL65MM2RmpB9C9RXYmwxCdlO3GnCuR6bJzKpalwfNnGfz5a30uL7S7Wvli8i/u2shOaJN+NPQR0+DIUiBlQFBvlsqT4OvszfQu1nJCNBqTE33uI7QIZ5HKneGpURb6DaQGuRP7gEyRMvhXrUZ/hW84ASXqpwdANJ/Q9zDd4xuOYxkNIfHa8cvnYaVIGfP+2Wfh7u8z96yYSkHt/4pt1Qp5Ep9Z/8XB0GMRpxKco16wx9wF1EMMjbSCTBx8is6D2UCsRCKotrA1KRbRvhRHF0y/BkAX6Wb0EvaY1Lh21dx/9krublD1r/+xCd1sWzlCtIyPmVCaVmizNiBOLz+wtrnvPSSg4chpgq0YALjH5TSWYIwIhvrwUmiEOgz69LF7jSNAqZ43ERAqtR692VAP7Os/O7CDykYINcEkFh9vjP

#### Updating State

In [32]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f19aea9-fff9-6e08-8000-ce7aa18cf48f", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19aebb-b7ca-681d-8001-e1540a31a77a'}}

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19aebb-b7ca-681d-8001-e1540a31a77a'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-18T10:01:07.869678+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19aea9-fff9-6e08-8000-ce7aa18cf48f'}}, tasks=(PregelTask(id='f025a7ad-3eae-a17f-3706-954baaeacc64', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza chef go to therapy?\n\nBecause he felt like nobody **kneaded** him, and everybody just wanted a **piece** of him!', 'extras': {'signature': 'ErYjCrMjARFNMg/o16DTJFitEq+L53843Soo6Rtc7o2a7sBSbZflTAKfaz4Mu/Y72VbYlaOxnQTv84AKUyIhpC10b7V4Nr8gDMEHCnWWyBg7WAH4L0ETSOxW0c14hD/5HND7u6VfFXEfG92pO

In [34]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19aea9-fff9-6e08-8000-ce7aa18cf48f"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'I was going to tell you a joke about pizza... \n\n...but it’s a bit too **cheesy**, and honestly, the **delivery** is terrible!',
   'extras': {'signature': 'Er8bCrwbARFNMg9Fp0fWDxwAcDMRb2ZclX8vezomaZyWPlwiGxrzvpBWrMGpyRxoXqF9J/6RBpwYokUUAf/bnMuy7R7KLC5SsnBX+tx95+X4uNTbf0Z1VDFBxKnP0Z7ngB5qZtgLRygc5yvZpAkqjY/nHgw6nI/WB32YCn3MV8BE4xroU6Slq1+Dq4kCfQZieSy07MhddafuTKhZ8FfUmRkC2Tfq+mlclr2ZJuBbfpxTkp5B5IvTPxMQ8c42ToTpmoE7+EQTUzvh3ie8zsigALQ8u6FfHPzL9LPQo7ho5EtQwYzvoPwNHkjILz7JYWwrTYDuufy/zCOeUBh+kNH+3ScWIvUvoLAoCivM8EcGxebv5LCTUBSBfMI0KA3Nnr4bNjLhfNqFCP5pfpAAAaKcStep3I7CpJsqX95xtf68X0/tNcqLI/Izt3xGZHjhOpRY95JySyaGMkT3+gWFtl83GgHDH/G2lxDmjQd27KmqVLRafjneaefyD+PdToqlbroXXkCMPOB3GCT/ei4+pXNS6LeSTuyiywzHPK1L9CeDBKtBTLhewm5giVsEy/36teyHbILT23GlBeCVSte7YwUn08IAeBM3qqkOZb/fGrIKsy3NbiK7XMVRlmsJneGSkdbI4Eb/9kuMLdedwycVmrzUjn9+kle0MAkvZKgww32ygQblBGZ4dFs1nw5dlBq5rZXaHZo544QVW2bj+Ykd6E0wBD+lUgfCuijhoF2OZzuQWu39KS2vqzgZuM9uFY1rAdDMD1J30wf2gX/ERT7E

In [38]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I was going to tell you a joke about pizza... \n\n...but it’s a bit too **cheesy**, and honestly, the **delivery** is terrible!', 'extras': {'signature': 'Er8bCrwbARFNMg9Fp0fWDxwAcDMRb2ZclX8vezomaZyWPlwiGxrzvpBWrMGpyRxoXqF9J/6RBpwYokUUAf/bnMuy7R7KLC5SsnBX+tx95+X4uNTbf0Z1VDFBxKnP0Z7ngB5qZtgLRygc5yvZpAkqjY/nHgw6nI/WB32YCn3MV8BE4xroU6Slq1+Dq4kCfQZieSy07MhddafuTKhZ8FfUmRkC2Tfq+mlclr2ZJuBbfpxTkp5B5IvTPxMQ8c42ToTpmoE7+EQTUzvh3ie8zsigALQ8u6FfHPzL9LPQo7ho5EtQwYzvoPwNHkjILz7JYWwrTYDuufy/zCOeUBh+kNH+3ScWIvUvoLAoCivM8EcGxebv5LCTUBSBfMI0KA3Nnr4bNjLhfNqFCP5pfpAAAaKcStep3I7CpJsqX95xtf68X0/tNcqLI/Izt3xGZHjhOpRY95JySyaGMkT3+gWFtl83GgHDH/G2lxDmjQd27KmqVLRafjneaefyD+PdToqlbroXXkCMPOB3GCT/ei4+pXNS6LeSTuyiywzHPK1L9CeDBKtBTLhewm5giVsEy/36teyHbILT23GlBeCVSte7YwUn08IAeBM3qqkOZb/fGrIKsy3NbiK7XMVRlmsJneGSkdbI4Eb/9kuMLdedwycVmrzUjn9+kle0MAkvZKgww32ygQblBGZ4dFs1nw5dlBq5rZXaHZo544QVW2bj+Ykd6E0wBD+lUgfCuijhoF2OZzuQWu39KS2vqzgZuM9uFY1rAdDMD

### Fault Tolerance

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))